# The whole pipeline, end to end

Chapters 2.2 to 2.8 each took one step slowly. This notebook runs all of them in
one pass, with no digressions, so you can see the shape of a complete linkage and
copy it as a starting point for your own.

It is deliberately short. A working linkage pipeline is not a large amount of
code; what takes the time is the judgement about what goes in it, which is what
the preceding chapters were for.

**One note.** This notebook re-trains the model rather than loading the saved
one, so its parameter estimates come from a fresh, unseeded *u* sampling. Step 6
checks how much difference that makes.

## Step 1 — Load

In [1]:
import unicodedata
from pathlib import Path

import pandas as pd
import phonetics

from splink import DuckDBAPI, Linker, SettingsCreator, block_on
import splink.comparison_library as cl

DATA = Path("../../data")
if not DATA.exists():
    DATA = Path("data")

fonasa = pd.read_csv(DATA / "fonasa_sample.csv", dtype=str)
suseso = pd.read_csv(DATA / "suseso_sample.csv", dtype=str)

TRUE_MATCHES = 4500
print(f"{len(fonasa):,} + {len(suseso):,} records")

30,000 + 27,000 records


## Step 2 — Clean, standardise, encode

Identical treatment of both sources. This is chapter 2.2 in one cell.

In [2]:
def basic_text_clean(s):
    return (s.astype("string").str.strip().str.upper()
            .str.replace(r"\s+", " ", regex=True))


def remove_accents(v):
    if pd.isna(v):
        return pd.NA
    return "".join(c for c in unicodedata.normalize("NFKD", str(v))
                   if not unicodedata.combining(c))


def standardise_name(s):
    c = basic_text_clean(s).map(remove_accents, na_action="ignore").astype("string")
    return (c.str.replace(r"[^A-ZN ]", "", regex=True)
             .str.replace(r"\s+", " ", regex=True).str.strip())


def dmeta_primary(v):
    if pd.isna(v) or v == "":
        return None
    try:
        return phonetics.dmetaphone(str(v))[0] or None
    except Exception:
        return None


def dmeta_list(v):
    if pd.isna(v) or v == "":
        return None
    try:
        return [c for c in phonetics.dmetaphone(str(v)) if c] or None
    except Exception:
        return None


for df in (fonasa, suseso):
    for col in ["nombre", "ap1", "ap2"]:
        df[f"{col}_clean"] = standardise_name(df[col])
        df[f"{col}_dm"] = df[f"{col}_clean"].map(dmeta_primary)
        df[f"{col}_dmeta"] = df[f"{col}_clean"].map(dmeta_list)
    df["sexo_clean"] = basic_text_clean(df["sexo"]).replace(
        {"HOMBRE": "M", "MUJER": "F", "": pd.NA})
    df["nac_clean"] = basic_text_clean(df["nacionalidad"])

COLS = ["unique_id", "nombre_clean", "ap1_clean", "ap2_clean", "sexo_clean",
        "nac_clean", "nombre_dm", "ap1_dm", "ap2_dm",
        "nombre_dmeta", "ap1_dmeta", "ap2_dmeta", "true_person_id"]
left, right = fonasa[COLS].copy(), suseso[COLS].copy()
print("prepared")

prepared


## Step 3 — Define the model

Blocking rules from chapter 2.5, comparisons from chapter 2.6.

In [3]:
blocking_rules = [
    block_on("nombre_clean", "ap1_clean"),
    block_on("ap1_clean", "ap2_clean"),
    block_on("ap1_dm", "ap2_dm"),
    block_on("nombre_dm", "ap1_dm"),
]

comparisons = [
    cl.NameComparison("nombre_clean", dmeta_col_name="nombre_dmeta")
        .configure(term_frequency_adjustments=True),
    cl.NameComparison("ap1_clean", dmeta_col_name="ap1_dmeta")
        .configure(term_frequency_adjustments=True),
    cl.NameComparison("ap2_clean", dmeta_col_name="ap2_dmeta")
        .configure(term_frequency_adjustments=True),
    cl.ExactMatch("sexo_clean"),
    cl.ExactMatch("nac_clean").configure(term_frequency_adjustments=True),
]

linker = Linker(
    input_table_or_tables=[left, right],
    settings=SettingsCreator(
        link_type="link_only",
        blocking_rules_to_generate_predictions=blocking_rules,
        comparisons=comparisons,
        unique_id_column_name="unique_id",
        retain_intermediate_calculation_columns=True,
        additional_columns_to_retain=["true_person_id"],
    ),
    db_api=DuckDBAPI(),
    input_table_aliases=["fonasa", "suseso"],
)
print("model defined")

model defined


Note that this uses **four** blocking rules, not the six of chapter 2.6. The
cumulative analysis there showed that the two rules built on the given-name
initial contributed no additional pairs, so they have been dropped. Same
candidate set, less work.

## Step 4 — Train

In [4]:
linker.training.estimate_probability_two_random_records_match(
    [block_on("nombre_clean", "ap1_clean", "ap2_clean")], recall=0.5)
linker.training.estimate_u_using_random_sampling(max_pairs=5e6)

for rule in [block_on("nombre_clean", "ap1_clean"),
             block_on("ap1_clean", "ap2_clean"),
             block_on("nombre_clean", "sexo_clean")]:
    linker.training.estimate_parameters_using_expectation_maximisation(rule)

print("trained")

Probability two random records match is estimated to be  2.69e-06.
This means that amongst all possible pairwise record comparisons, one in 372,242.65 are expected to match.  With 810,000,000 total possible comparisons, we expect a total of around 2,176.00 matching pairs
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nombre_clean (no m values are trained).
    - ap1_clean (no m values are trained).
    - ap2_clean (no m values are trained).
    - sexo_clean (no m values are trained).
    - nac_clean (no m values are trained).

----- Starting EM training session -----

Estimating the m probabilities of the model by blocking on:
(l."nombre_clean" = r."nombre_clean") AND (l."ap1_clean" = r."ap1_clean")

Parameter estimates will be made for the following comparison(s):
    - ap2_clean
    - sexo_clean
    - nac_clean

Parameter estimates cannot be made for the f

trained


## Step 5 — Predict, threshold, cluster

In [5]:
THRESHOLD = 0.9

df_predict = linker.inference.predict()
predictions = df_predict.as_pandas_dataframe()

clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=THRESHOLD
).as_pandas_dataframe()

linked = predictions[predictions["match_probability"] >= THRESHOLD]

print(f"candidate pairs : {len(predictions):,}")
print(f"linked pairs    : {len(linked):,}")
print(f"clusters        : {clusters['cluster_id'].nunique():,}")

Blocking time: 0.03 seconds
Predict time: 0.09 seconds
Completed iteration 1, num edges remaining to process: 0


candidate pairs : 8,868
linked pairs    : 1,264
clusters        : 55,736


## Step 6 — Evaluate

In [6]:
tp = int((linked["true_person_id_l"] == linked["true_person_id_r"]).sum())
fp = len(linked) - tp
precision = tp / len(linked)
recall = tp / TRUE_MATCHES

sizes = clusters["cluster_id"].value_counts()

print(f"precision                 : {precision:.4f}")
print(f"recall (all true matches) : {recall:.4f}")
print(f"false matches             : {fp:,}")
print(f"clusters larger than two  : {(sizes > 2).sum():,}")

precision                 : 0.9960
recall (all true matches) : 0.2798
false matches             : 5
clusters larger than two  : 0


In [7]:
# Chapter 2.7 ran the same threshold against the model saved in chapter 2.6
REF = {"precision": 0.9961, "recall": 0.2804, "linked_pairs": 1267}

print("Same pipeline, freshly trained, versus the saved model of chapter 2.6:")
print(f"  precision : {precision:.4f}  (chapter 2.7: {REF['precision']:.4f}, "
      f"difference {precision - REF['precision']:+.4f})")
print(f"  recall    : {recall:.4f}  (chapter 2.7: {REF['recall']:.4f}, "
      f"difference {recall - REF['recall']:+.4f})")
print(f"  pairs     : {len(linked):,}      (chapter 2.7: {REF['linked_pairs']:,})")

Same pipeline, freshly trained, versus the saved model of chapter 2.6:
  precision : 0.9960  (chapter 2.7: 0.9961, difference -0.0001)
  recall    : 0.2798  (chapter 2.7: 0.2804, difference -0.0006)
  pairs     : 1,264      (chapter 2.7: 1,267)


Whatever the printed differences are, note that they are **not zero by
construction**, and that re-running this notebook will produce a slightly
different set. They come from one source: `estimate_u_using_random_sampling()`
draws a fresh random sample each time, and nothing seeds it.

How much that matters depends on the sample size. Below, the same estimation is
run three times at each of two sample sizes.

In [8]:
import contextlib
import io


def u_for_exact_name_level(max_pairs):
    # Estimate u only, at a given sample size, and return one parameter.
    lk = Linker(
        input_table_or_tables=[left, right],
        settings=SettingsCreator(
            link_type="link_only",
            blocking_rules_to_generate_predictions=blocking_rules,
            comparisons=comparisons,
            unique_id_column_name="unique_id",
        ),
        db_api=DuckDBAPI(),
        input_table_aliases=["fonasa", "suseso"],
    )
    lk.training.estimate_u_using_random_sampling(max_pairs=max_pairs)
    comp = lk._settings_obj.comparisons[0]
    level = [l for l in comp.comparison_levels
             if "Exact match" in str(l.label_for_charts)][0]
    return level.u_probability


for max_pairs in [1e4, 1e6]:
    values = []
    for _ in range(3):
        with contextlib.redirect_stdout(io.StringIO()):
            values.append(u_for_exact_name_level(max_pairs))
    print(f"max_pairs={max_pairs:.0e}:  "
          f"u = {[f'{v:.6f}' for v in values]}  "
          f"spread {max(values) - min(values):.6f}")

----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nombre_clean (no m values are trained).
    - ap1_clean (no m values are trained).
    - ap2_clean (no m values are trained).
    - sexo_clean (no m values are trained).
    - nac_clean (no m values are trained).
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nombre_clean (no m values are trained).
    - ap1_clean (no m values are trained).
    - ap2_clean (no m values are trained).
    - sexo_clean (no m values are trained).
    - nac_clean (no m values are trained).
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nombre_clean (no m values are trained).
   

max_pairs=1e+04:  u = ['0.000418', '0.000506', '0.000169']  spread 0.000337



Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nombre_clean (no m values are trained).
    - ap1_clean (no m values are trained).
    - ap2_clean (no m values are trained).
    - sexo_clean (no m values are trained).
    - nac_clean (no m values are trained).
You are using the default value for `max_pairs`, which may be too small and thus lead to inaccurate estimates for your model's u-parameters. Consider increasing to 1e8 or 1e9, which will result in more accurate estimates, but with a longer run time.
----- Estimating u probabilities using random sampling -----

Estimated u probabilities using random sampling

Your model is not yet fully trained. Missing estimates for:
    - nombre_clean (no m values are trained).
    - ap1_clean (no m values are trained).
    - ap2_clean (no m values are trained).
    - sexo_clean (no m values are trained).
    - nac_clean (no m values are trained).
You are using the default value

max_pairs=1e+06:  u = ['0.000164', '0.000126', '0.000117']  spread 0.000047


At the small sample size the estimate for a single comparison level varies by a
factor of several between runs. A hundred times more pairs tightens it by roughly
an order of magnitude — but it still varies, and it never settles on one value.

Why, then, do the headline precision and recall barely move? Because the model's
*ranking* of pairs is robust to small changes in individual parameters: a pair
that scores far above the threshold keeps scoring far above it. The instability
shows up where the score distribution is thin, which is exactly around a
threshold chosen by optimising something — as chapter 2.8's F1-optimal threshold
was.

So the lesson is not "use a big sample and stop worrying". It is that **an
unseeded estimation step makes results irreproducible by default**, and the
reliable fix is to train once, save the model, and version it alongside the code.
Section 3 treats this as a reproducibility requirement rather than a nicety.

## Step 7 — Export

In [9]:
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

(linked[["unique_id_l", "unique_id_r", "match_probability", "match_weight"]]
 .rename(columns={"unique_id_l": "fonasa_record", "unique_id_r": "suseso_record"})
 .to_csv(OUT / "end-to-end-linked-pairs.csv", index=False))

linker.misc.save_model_to_json(str(OUT / "end-to-end-model.json"), overwrite=True)

print("written:", *[f.name for f in sorted(OUT.glob("end-to-end-*"))])

written: end-to-end-linked-pairs.csv end-to-end-model.json


## The run, summarised

Something like this table belongs in your project documentation. It is the
minimum a reader needs to judge the result, and Section 3 develops it into a
publishable quality statement.

In [10]:
pd.DataFrame({
    "item": [
        "Left source", "Right source", "Link type",
        "Identifying variables",
        "Cleaning", "Phonetic encoding",
        "Blocking rules", "Candidate pairs", "Pairs scored per possible pair",
        "Comparisons", "Parameter estimation",
        "Threshold (match probability)",
        "Linked pairs", "Clusters", "Clusters larger than two",
        "Precision (against known labels)",
        "Recall (against all true matches)",
        "Recall ceiling imposed by blocking",
    ],
    "value": [
        f"synthetic health-insurance register ({len(left):,} records)",
        f"synthetic social-security register ({len(right):,} records)",
        "link_only (1:1 expected)",
        "given name, first surname, second surname, sex, nationality",
        "uppercase, strip accents and punctuation, collapse whitespace",
        "Double Metaphone on all three name fields",
        f"{len(blocking_rules)} rules, unioned",
        f"{len(predictions):,}",
        f"{len(predictions)/(len(left)*len(right)):.2e}",
        "fuzzy name comparison with phonetic level; term-frequency adjusted",
        "lambda from deterministic rules (recall=0.5); u by random sampling; m by EM over 3 training rules",
        f"{THRESHOLD}",
        f"{len(linked):,}",
        f"{clusters['cluster_id'].nunique():,}",
        f"{(sizes > 2).sum():,}",
        f"{precision:.4f}",
        f"{recall:.4f}",
        "0.4182 (measured in chapter 2.5)",
    ],
})

,item,value
0,Left source,"synthetic health-insurance register (30,000 re..."
1,Right source,"synthetic social-security register (27,000 rec..."
2,Link type,link_only (1:1 expected)
3,Identifying variables,"given name, first surname, second surname, sex..."
4,Cleaning,"uppercase, strip accents and punctuation, coll..."
5,Phonetic encoding,Double Metaphone on all three name fields
6,Blocking rules,"4 rules, unioned"
7,Candidate pairs,"8,868"
8,Pairs scored per possible pair,1.09e-05
9,Comparisons,fuzzy name comparison with phonetic level; ter...


## Adapting this to your own data

Five things to change, in order of how much thought each needs.

1. **The columns.** Replace the five identifying fields with yours. If you have a
   date of birth, add it — it is usually the single most valuable comparison
   field, and its absence here is what makes this a hard example.
2. **The cleaning.** The functions in step 2 suit Latin-script names with two
   surnames. Names in your setting may need transliteration, different
   punctuation handling, or no accent stripping at all.
3. **The blocking rules.** Do not copy these. Design them against your own data
   and measure pair completeness, as chapter 2.5 sets out. This is where most of
   the achievable recall is decided.
4. **The `recall=0.5` assumption** in the λ estimate. Justify it or test its
   effect.
5. **The threshold.** Choose it from your error preference and your measured
   error rates, not from this notebook.

What you should not need to change is the shape: prepare, block, compare, train,
predict, threshold, cluster, evaluate, document. That sequence is the same for
every linkage project.